In [2]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, StandardScaler, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Create a SparkSession
spark = SparkSession.builder.appName("LogisticRegressionPipeline").getOrCreate()

# Sample PySpark DataFrame
data = [
    ("Male", "USA", 25, 1),
    ("Female", "Canada", 30, 0),
    ("Male", "USA", 40, 1),
    ("Female", "Mexico", 22, 0),
    ("Male", "Canada", 35, 1),
    ("Female", "USA", 28, 0),
]
columns = ["gender", "country", "age", "label"]
df = spark.createDataFrame(data, columns)

# Identify categorical and numerical columns
categorical_cols = ["gender", "country"]
numerical_col_to_scale = "age"
label_col = "label"

# --- Pipeline Stages ---

# 1. StringIndexer for categorical features
indexers = [StringIndexer(inputCol=col, outputCol=col + "_index")
            for col in categorical_cols]

# 2. OneHotEncoder for indexed categorical features
encoders = [OneHotEncoder(inputCol=indexer.getOutputCol(),
                          outputCol=indexer.getOutputCol() + "_encoded")
            for indexer in indexers]

# 3. VectorAssembler for the numerical feature BEFORE scaling
numerical_assembler = VectorAssembler(inputCols=[numerical_col_to_scale],
                                       outputCol=numerical_col_to_scale + "_vector")

# 4. StandardScaler for the vector of the numerical feature
scaler = StandardScaler(inputCol=numerical_col_to_scale + "_vector",
                        outputCol=numerical_col_to_scale + "_scaled")

# 5. VectorAssembler to combine all features into a single vector for the model
assembler_inputs = [encoder.getOutputCol() for encoder in encoders] + [scaler.getOutputCol()]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

# 6. Logistic Regression model
lr = LogisticRegression(labelCol=label_col, featuresCol="features")

# --- Create the Pipeline ---
pipeline = Pipeline(stages=indexers + encoders + [numerical_assembler, scaler, assembler, lr])

# --- Fit the pipeline to the data ---
pipeline_model = pipeline.fit(df)

# --- Make predictions on new data ---
new_data = [
    ("Female", "USA", 32),
    ("Male", "Canada", 27),
]
new_columns = ["gender", "country", "age"]
new_df = spark.createDataFrame(new_data, new_columns)

predictions = pipeline_model.transform(new_df)
predictions.select("gender", "country", "age", "features", "prediction").show()



25/05/08 19:08:45 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/morgan/miniconda3/envs/venv39/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/morgan/miniconda3/envs/venv39/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/morgan/miniconda3/envs/venv39/lib/python3.9/socket.py", line 716, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 